In [1]:
import pandas as pd
data = {"order_id": [1,2,3,4,5,6],"customer_id": [101,102,101,103,104,105],
"delivery_date": ["2026-05-01","2026-05-08","2026-05-09","2026-05-10","2026-05-12","2026-05-13"],
"issue_type": ["Late Shipment","No Issue","Warehouse Delay","Weather Delay","No Issue","Transport Delay"]}
df = pd.DataFrame(data)
df.to_csv("orders.csv",index=False)
print("orders.csv created")

orders.csv created


In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("orders.csv")
print(df)

   order_id  customer_id delivery_date       issue_type
0         1          101    2026-05-01    Late Shipment
1         2          102    2026-05-08         No Issue
2         3          101    2026-05-09  Warehouse Delay
3         4          103    2026-05-10    Weather Delay
4         5          104    2026-05-12         No Issue
5         6          105    2026-05-13  Transport Delay


In [3]:
df['delivery_date'] = pd.to_datetime(df['delivery_date'])


In [4]:
df = df.dropna()
df['delay_days'] = (pd.Timestamp.today() -df['delivery_date']).dt.days

In [5]:
df['delayed'] = np.where(df['delay_days'] > 0,1,0)
print("\nProcessed Data")
print(df)


Processed Data
   order_id  customer_id delivery_date       issue_type  delay_days  delayed
0         1          101    2026-05-01    Late Shipment          13        1
1         2          102    2026-05-08         No Issue           6        1
2         3          101    2026-05-09  Warehouse Delay           5        1
3         4          103    2026-05-10    Weather Delay           4        1
4         5          104    2026-05-12         No Issue           2        1
5         6          105    2026-05-13  Transport Delay           1        1


In [6]:
delay_summary = df.groupby('customer_id')['delayed'].sum().sort_values(ascending=False)
print("\nTop Delayed Customers")
print(delay_summary)


Top Delayed Customers
customer_id
101    2
102    1
103    1
104    1
105    1
Name: delayed, dtype: int64


In [7]:
issue_summary = df['issue_type'].value_counts()
print("\nMost Common Delivery Issues")
print(issue_summary)


Most Common Delivery Issues
issue_type
No Issue           2
Late Shipment      1
Warehouse Delay    1
Weather Delay      1
Transport Delay    1
Name: count, dtype: int64


In [8]:
!pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder .appName("CustomerOrderAnalysis") .getOrCreate()
print("Spark Session Created")

Spark Session Created


In [9]:
orders_df = spark.read.csv("orders.csv",header=True,inferSchema=True)
orders_df.show()

+--------+-----------+-------------+---------------+
|order_id|customer_id|delivery_date|     issue_type|
+--------+-----------+-------------+---------------+
|       1|        101|   2026-05-01|  Late Shipment|
|       2|        102|   2026-05-08|       No Issue|
|       3|        101|   2026-05-09|Warehouse Delay|
|       4|        103|   2026-05-10|  Weather Delay|
|       5|        104|   2026-05-12|       No Issue|
|       6|        105|   2026-05-13|Transport Delay|
+--------+-----------+-------------+---------------+



In [10]:
customer_data = [(101,"Arun","South"),(102,"Priya","North"),(103,"Karthik","East"),
(104,"Meena","West"),(105,"Rahul","South")]
customers_df = spark.createDataFrame(customer_data,["customer_id","customer_name","region"])
customers_df.show()

+-----------+-------------+------+
|customer_id|customer_name|region|
+-----------+-------------+------+
|        101|         Arun| South|
|        102|        Priya| North|
|        103|      Karthik|  East|
|        104|        Meena|  West|
|        105|        Rahul| South|
+-----------+-------------+------+



In [11]:
joined_df = orders_df.join(customers_df,on="customer_id",how="inner")
joined_df.show()

+-----------+--------+-------------+---------------+-------------+------+
|customer_id|order_id|delivery_date|     issue_type|customer_name|region|
+-----------+--------+-------------+---------------+-------------+------+
|        101|       3|   2026-05-09|Warehouse Delay|         Arun| South|
|        101|       1|   2026-05-01|  Late Shipment|         Arun| South|
|        102|       2|   2026-05-08|       No Issue|        Priya| North|
|        103|       4|   2026-05-10|  Weather Delay|      Karthik|  East|
|        104|       5|   2026-05-12|       No Issue|        Meena|  West|
|        105|       6|   2026-05-13|Transport Delay|        Rahul| South|
+-----------+--------+-------------+---------------+-------------+------+



In [12]:
delayed_df = joined_df.filter(col("issue_type") != "No Issue")
delayed_df.show()

+-----------+--------+-------------+---------------+-------------+------+
|customer_id|order_id|delivery_date|     issue_type|customer_name|region|
+-----------+--------+-------------+---------------+-------------+------+
|        101|       3|   2026-05-09|Warehouse Delay|         Arun| South|
|        101|       1|   2026-05-01|  Late Shipment|         Arun| South|
|        103|       4|   2026-05-10|  Weather Delay|      Karthik|  East|
|        105|       6|   2026-05-13|Transport Delay|        Rahul| South|
+-----------+--------+-------------+---------------+-------------+------+



In [13]:
region_summary = delayed_df.groupBy("region").count()
region_summary.show()

+------+-----+
|region|count|
+------+-----+
| South|    3|
|  East|    1|
+------+-----+



In [14]:
region_summary.write.mode("overwrite").csv("delayed_orders_by_region")
print("CSV Output Saved")

CSV Output Saved


In [15]:
!ls delayed_orders_by_region

part-00000-e8c93466-c4b8-4ec7-974b-d5df964af0d8-c000.csv  _SUCCESS
